Loading raw pdf

In [20]:
import pymupdf
import re
import json
import langchain
doc = pymupdf.open(r"../data/raw/raw.pdf")
print("Total pages:", len(doc))

Total pages: 114


Cleaning the text

In [21]:
def clean_text(text: str) -> str:
    """Collapse excessive whitespace/newlines from OCR'd text into single spaces."""
    text = re.sub(r"\s+", " ", text)
    return text.strip()

Loading all pages into list

In [22]:
def load_all_pages(doc):
    pages = []
    for i, page in enumerate(doc):
        raw = page.get_text()
        pages.append({
            "page_no": i + 1,
            "text": clean_text(raw)
        })
    return pages

pages = load_all_pages(doc)
print("Loaded pages:", len(pages))
print(pages[20])   # sanity check - print a random page's cleaned text

Loaded pages: 114
{'page_no': 21, 'text': "18 456: 2000 Indian Standard PLAIN AND REINFORCED CONCRETE - CODE OFPRACTICE ( Fourth Revision ) FOREWORD ThisIndianStandard (Fourth Revision) wasadopted bytheBureau ofIndian Standards, afterthedraftfinalized . bytheCementandConcrete SectionalCommittee hadbeenapproved bytheCivilEngineering Division Council. Thisstandardwasfirstpublished in 1953 underthetitle 'Codeof practice forplainandreinforced concrete for general buildingconstruction' and subsequently revised in 1957. The code was further revised in 1964and published undermodified title 'Codeof practice forplainandreinforced concrete',thusenlarging the scopeof useof thiscodeto structures otherthangeneral building construction also. Thethirdrevision was published in 1978, and it includedlimitstateapproach to design. This is thefourth revision of the standard. Thisrevision wastakenupwitha viewto keeping abreast withtherapiddevelopment inthefieldofconcrete technology and to bringin furthermod

In [23]:
for idx in [0, 20, 60, 100]:
    print(f"--- page_no {pages[idx]['page_no']} ---")
    print(pages[idx]["text"][:300])
    print()

--- page_no 1 ---
Disclosure to Promote the Right To Information Whereas the Parliament of India has set out to provide a practical regime of right to information for citizens to secure access to information under the control of public authorities, in order to promote transparency and accountability in the working of

--- page_no 21 ---
18 456: 2000 Indian Standard PLAIN AND REINFORCED CONCRETE - CODE OFPRACTICE ( Fourth Revision ) FOREWORD ThisIndianStandard (Fourth Revision) wasadopted bytheBureau ofIndian Standards, afterthedraftfinalized . bytheCementandConcrete SectionalCommittee hadbeenapproved bytheCivilEngineering Division 

--- page_no 61 ---
15456:2000 under consideration. 1ft no case shall the spacinl exceed300mm. 26.5.1.6 Minim"". ,ltearrein/orreNnt Minimum shearreinforcement in the formof stinups shall beprovidedsuch that: where AI. • total cross-sectional areaof stirruplop effective in shear. '. • stirrup SPacinl alonl tho Ieft8th o

--- page_no 101 ---
IS 456: 2000 ANNEX

Garbled OCR words exsit, it will be taken care later on

Chunking 

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]  
)

def chunk_document_langchain(pages):
    chunks = []
    chunk_id = 0
    for p in pages:
        splits = splitter.split_text(p["text"])
        for s in splits:
            chunks.append({
                "chunk_id": f"chunk_{chunk_id:04d}",
                "page_no": p["page_no"],
                "text": s.strip()
            })
            chunk_id += 1
    return chunks

In [25]:
chunks = chunk_document_langchain(pages)
total_chunks = len(chunks)
print("Total chunks:", total_chunks)

Total chunks: 507


In [26]:
chunks

[{'chunk_id': 'chunk_0000',
  'page_no': 1,
  'text': 'Disclosure to Promote the Right To Information Whereas the Parliament of India has set out to provide a practical regime of right to information for citizens to secure access to information under the control of public authorities, in order to promote transparency and accountability in the working of every public authority, and whereas the attached publication of the Bureau of Indian Standards is of particular interest to the public, particularly disadvantaged communities and those engaged in the pursuit of education and knowledge, the attached public safety standard is made available to promote the timely dissemination of this information in an accurate manner to the public'},
 {'chunk_id': 'chunk_0001',
  'page_no': 1,
  'text': ". इंटरनेट मानक “!ान$ एकन' भारतका+नम-ण” Satyanarayan Gangaram Pitroda “Invent a New India Using Knowledge” “प0रा1 कोछोडन' 5 तरफ” Jawaharlal Nehru “Step Out From the Old to the New” “जान1 का अ+धकार, जी1 का 

In [27]:
import random
for c in random.sample(chunks, 5):
    print(f"[{c['chunk_id']}] page {c['page_no']} | len={len(c['text'])}")
    print(c["text"][:400])
    print("---")

[chunk_0316] page 69 | len=716
. IKe' doesnotsatisfy(a).thepositive design moments forthepanelsball bemultiplied by the coefficient P. given by the following equation: b) Eachsuchframe maybeanalyzed initsentirety, or, for vertical loading, eachfloor thereofand the roof may be analyzed separately with its columns being assumed fixed at their remote ends. Whereslabsarethusanalyzed separately, it maybe assumed in determining the b
---
[chunk_0087] page 29 | len=677
. the cement content, the relative humidity and the size of sections. The value of coefficient of thermal expansion forconcretewith different aggregates maybe takenas below: where Ec is the shorttenn staticmodulus of elasticityin N/mm2• Actual measured values may differ by ± 20 percent fromthe values obtained fromthe aboveexpression. 6.2.4 Shrinlcage The total shrinkage of concrete depeRds upon th
---
[chunk_0306] page 67 | len=476
IS 456 : 2000 ....-~........--r- CRITICAL SECTION 'OR SHUR IMMfDIATlLY ADJACENT TO COLUM 12 B SLA

In [ ]:
import json
with open("../data/processed/chunks.json", "w") as f:
    json.dump(chunks, f, indent=2)
print("Saved", len(chunks), "chunks")

Saved 507 chunks


In [1]:
import json
from sentence_transformers import SentenceTransformer

with open("../data/processed/chunks.json") as f:
    chunks = json.load(f)

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("Model loaded. Embedding dim:", model.get_sentence_embedding_dimension())

d:\Acads\Analystics and software\Projects\2. IS 456 RAG chatbot - retrieval-augmented QA over the Indian Standard code for reinforced concrete\LLM-Based-IS-456-QA-Agent\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3437.09it/s]


Model loaded. Embedding dim: 384


C:\Users\mdyas\AppData\Local\Temp\ipykernel_10064\3220100017.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded. Embedding dim:", model.get_sentence_embedding_dimension())


In [2]:
texts = [c["text"] for c in chunks]
embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)
print(embeddings.shape)  # should be (507, 384) for bge-small

Batches: 100%|██████████| 16/16 [01:31<00:00,  5.70s/it]

(507, 384)


In [3]:
import chromadb

client = chromadb.PersistentClient(path="../data/processed/chroma_db")

collection = client.get_or_create_collection(
    name="is456_chunks",
    metadata={"hnsw:space": "cosine"}  # cosine similarity for search
)
print("Collection ready:", collection.name)

Collection ready: is456_chunks


In [4]:
collection.add(
    ids=[c["chunk_id"] for c in chunks],
    embeddings=embeddings.tolist(),
    documents=[c["text"] for c in chunks],
    metadatas=[{"page_no": c["page_no"]} for c in chunks]
)
print("Added", collection.count(), "chunks to ChromaDB")

Added 507 chunks to ChromaDB


In [5]:
def search(query, top_k=5):
    query_embedding = model.encode([query])[0].tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return results

results = search("minimum grade of concrete for severe exposure condition")
for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"page {meta['page_no']} | distance {dist:.4f}")
    print(doc[:300])
    print("---")

page 21 | distance 0.1678
. The table on 'Environmental Exposure Conditions' has been modified to include 'very severe' and 'extreme' exposure conditions. This clause also covers requirements for shape and size of member, depthofconcretecover, concretequality, requirement againstexposure toaggressive chemical andsulphate att
---
page 21 | distance 0.1737
. e) It has been recommended that minimum grade of concrete shall be not less than M 20 in reinforced concrete work (seealso6.1.3). f) The formula for estimation of modulus of elasticity of concrete hasbeenrevised. g) In the absenceof propercorrelation between compacting factor, vee-bee time and slu
---
page 31 | distance 0.2041
. Appropriate valuesforminimum cementcontentand the maximum free weler-cemenl ratio are given in Table S for different exposure conditions. The minimum cement content and maximum water-cementratio apply to 20mm nominal maximum sizeaggresate. Forothersizesof aggregate they shouldbechanged81 givenin T
---
page 31

In [6]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")  # adjust path as needed
api_key = os.getenv("GOOGLE_API_KEY")
print("Key loaded:", api_key is not None)

genai.configure(api_key=api_key)
gemini_model = genai.GenerativeModel("gemini-2.0-flash")

def generate_answer(query, top_k=5):
    search_results = search(query, top_k=top_k)
    retrieved_chunks = [
        {"text": doc, "page_no": meta["page_no"]}
        for doc, meta in zip(search_results["documents"][0], search_results["metadatas"][0])
    ]

    prompt = build_prompt(query, retrieved_chunks)

    response = gemini_model.generate_content(
        prompt,
        generation_config={"temperature": 0.1}
    )
    return response.text, retrieved_chunks

d:\Acads\Analystics and software\Projects\2. IS 456 RAG chatbot - retrieval-augmented QA over the Indian Standard code for reinforced concrete\LLM-Based-IS-456-QA-Agent\venv\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Key loaded: True


C:\Users\mdyas\AppData\Local\Temp\ipykernel_10064\102412524.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [7]:
def build_prompt(query, retrieved_chunks):
    context = "\n\n".join(
        f"[Page {c['page_no']}]\n{c['text']}"
        for c in retrieved_chunks
    )
    prompt = f"""You are an assistant answering questions about IS 456:2000, the Indian Standard code for Plain and Reinforced Concrete.

Use ONLY the context below to answer the question. If the context doesn't contain enough information to answer, say so clearly - do not make up information.

Always cite the page number(s) your answer is based on.

Context:
{context}

Question: {query}

Answer:"""
    return prompt

In [8]:
from dotenv import load_dotenv
import os

loaded = load_dotenv()
print("dotenv loaded:", loaded)
print("Key found:", os.getenv("GOOGLE_API_KEY") is not None)

dotenv loaded: True
Key found: True


In [11]:
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()
client_groq = Groq(api_key=os.getenv("GROQ_API_KEY"))

def generate_answer(query, top_k=5, model_name="llama-3.1-8b-instant"):
    search_results = search(query, top_k=top_k)
    retrieved_chunks = [
        {"text": doc, "page_no": meta["page_no"]}
        for doc, meta in zip(search_results["documents"][0], search_results["metadatas"][0])
    ]
    prompt = build_prompt(query, retrieved_chunks)
    response = client_groq.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
    )
    return response.choices[0].message.content, retrieved_chunks

In [14]:
answer, chunks_used = generate_answer("What is the formula for calculating the ultimate moment of resistance?")
print(answer)

The formula for calculating the ultimate moment of resistance is given in different scenarios in Annex G of IS 456: 2000.

For a rectangular section without compression reinforcement, the ultimate moment of resistance (Mu) is given by:

a) Mu = 0.87 f y (Astir) d - (bdfek) if the value of xuld is less than the limiting value (see Note below 38.1).

b) Mu = 0.36 (1 - 0.42 Xu, max) (Astir) d^2 / ek for a rectangular section without compression reinforcement.

c) For a flanged section, the ultimate moment of resistance (Mu) is given by:

- M = 0.36 (1 - 0.42 Xu, max) (Astir) d^2 / ek + 0.45fek (bf - bw)Df (d - pt) if the ratio Dr/d does not exceed 0.2.
- M = 0.36 (1 - 0.42 Xu, max) (Astir) d^2 / ek + 0.45fek (bf - bw)Df (d - pt) + 0.4fek (bwd^2) if the ratio Dr/d exceeds 0.2.

For a section with compression reinforcement, the ultimate moment of resistance (Mu) is given by:

Mu = Mu, u + Mu, compr, where Mu, u is the ultimate moment of resistance without compression reinforcement and Mu, c